# TP2 : Reconocimiento de Actividades Humanas con Machine Learning

## Contexto general

En este trabajo práctico vamos a abordar un problema clásico de **clasificación supervisada**: predecir la actividad que está realizando una persona a partir de mediciones obtenidas por sensores de un smartphone ([documentación del dataset](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones)).

Este tipo de problema aparece en muchas aplicaciones reales de ingeniería y ciencia de datos, por ejemplo:

- monitoreo de actividad física,
- salud digital,
- sistemas de asistencia,
- interfaces inteligentes,
- dispositivos *wearables* y móviles.

La idea central es construir un flujo de trabajo completo de *machine learning*, desde la exploración de los datos hasta la evaluación de distintos modelos de clasificación.

## Objetivos del TP

A lo largo de este notebook se espera que puedan:

- comprender la estructura general del problema,
- explorar y analizar el conjunto de datos,
- preparar las variables para el modelado,
- aplicar técnicas de reducción de dimensionalidad (PCA),
- entrenar distintos clasificadores (regresion logistica, arbol de decision y random forrest),
- comparar resultados y discutir ventajas y limitaciones de cada enfoque.

## Consigna general

No se busca solamente ejecutar código, sino también **interpretar** qué se está haciendo en cada etapa y **justificar** las decisiones tomadas.

En particular, al finalizar el TP deberían poder responder preguntas como:

- ¿qué tipo de problema de aprendizaje automático estamos resolviendo?
- ¿qué información contienen las variables de entrada?
- ¿por qué ciertas transformaciones son útiles para algunos modelos y no para otros?
- ¿cómo comparar modelos más allá de una única métrica?


### **Consignas presentacion:** La presentación debe tener un máximo de 4 diapositivas y una duración aproximada de **5 minutos**. La estructura de las diapositivas debe ser la siguiente:
  1. Presentación del grupo y del problema a resolver.
  2. Datos, pre-procesamiento y PCA.
  3. Modelado.
  4. Resultados/Conclusiones.

Durante la presentación deberán explicar claramente las siguientes etapas del trabajo:
- Análisis exploratorio de datos (EDA):
Qué observaron en los datos: estructura del dataset, variables importantes, valores faltantes, distribuciones, posibles outliers y relaciones entre variables.

- PCA : Nuevas variables, varianza acumulada y cantidad de variables.

- Modelos regresion logistica, Árbol de Decisión y Random Forrest:
Qué modelos probaron, por qué los eligieron y cómo ajustaron sus parámetros o configuraciones para mejorar el rendimiento.

- Evaluación y testeo del modelo:
Cómo evaluaron el modelo, qué métricas utilizaron y qué resultados obtuvieron en el conjunto de test.

El objetivo de esta presentación es mostrar que realmente comprendieron el proceso que realizaron y que pueden explicarlo de manera clara.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


In [ ]:
# Cargar datasets de train y test
df_train = pd.read_csv('https://raw.githubusercontent.com/juanserrano90/SIA_itba/refs/heads/main/TP2/train.csv')
df_test = pd.read_csv('https://raw.githubusercontent.com/juanserrano90/SIA_itba/refs/heads/main/TP2/test.csv')

### Descripción del conjunto de datos

El dataset utilizado corresponde a un problema de **reconocimiento de actividad humana**. Cada observación representa una ventana de señales registradas por sensores inerciales de un teléfono móvil, y la variable objetivo indica la actividad realizada por la persona.

En el conjunto de datos aparecen dos tipos de información importantes:

- una columna con la **actividad** a predecir,
- una columna que identifica al **sujeto**,
- un conjunto grande de variables numéricas derivadas de las señales medidas por los sensores.

### Observación importante

En este TP el objetivo principal es predecir la variable **`Activity`**.  
La columna **`subject`** no describe directamente el movimiento, sino la identidad del participante. *Por eso conviene pensar críticamente si debe utilizarse o no como predictor.*

## Hoja de ruta del análisis

El trabajo seguirá, de manera general, los siguientes pasos:

1. inspección inicial de los datos,
2. análisis exploratorio,
3. preparación de variables,
4. reducción de dimensionalidad,
5. entrenamiento de modelos,
6. evaluación y comparación de resultados.

In [ ]:
df_train.describe()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject
count,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,...,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000,7352.000000
mean,0.274488,-0.017695,-0.109141,-0.605438,-0.510938,-0.604754,-0.630512,-0.526907,-0.606150,-0.468604,...,-0.307009,-0.625294,0.008684,0.002186,0.008726,-0.005981,-0.489547,0.058593,-0.056515,17.413085
std,0.070261,0.040811,0.056635,0.448734,0.502645,0.418687,0.424073,0.485942,0.414122,0.544547,...,0.321011,0.307584,0.336787,0.448306,0.608303,0.477975,0.511807,0.297480,0.279122,8.975143
min,-1.000000,-1.000000,-1.000000,-1.000000,-0.999873,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-0.995357,-0.999765,-0.976580,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,1.000000
25%,0.262975,-0.024863,-0.120993,-0.992754,-0.978129,-0.980233,-0.993591,-0.978162,-0.980251,-0.936219,...,-0.542602,-0.845573,-0.121527,-0.289549,-0.482273,-0.376341,-0.812065,-0.017885,-0.143414,8.000000
50%,0.277193,-0.017219,-0.108676,-0.946196,-0.851897,-0.859365,-0.950709,-0.857328,-0.857143,-0.881637,...,-0.343685,-0.711692,0.009509,0.008943,0.008735,-0.000368,-0.709417,0.182071,0.003181,19.000000
75%,0.288461,-0.010783,-0.097794,-0.242813,-0.034231,-0.262415,-0.292680,-0.066701,-0.265671,-0.017129,...,-0.126979,-0.503878,0.150865,0.292861,0.506187,0.359368,-0.509079,0.248353,0.107659,26.000000
max,1.000000,1.000000,1.000000,1.000000,0.916238,1.000000,1.000000,0.967664,1.000000,1.000000,...,0.989538,0.956845,1.000000,1.000000,0.998702,0.996078,1.000000,0.478157,1.000000,30.000000


# 1. Análisis Exploratorio de Datos (EDA)
### Preguntas orientadoras

Mientras ejecutan las celdas, intenten responder:

a) ¿Cuántas observaciones y cuántas variables hay? \
b) ¿Todas las variables predictoras son numéricas?\
c) ¿Hay datos faltantes?\
d) ¿Las clases parecen estar balanceadas o desbalanceadas?\
e) ¿Qué implicancias podría tener esto para el entrenamiento?\
f) ¿Cuántos sujetos únicos hay?

El análisis exploratorio no solo sirve para “mirar” los datos: también ayuda a anticipar posibles decisiones de preprocesamiento y modelado.

In [ ]:
# Código

# 2. Preparación de los datos

### Guía orientadora:
* Codificar la variable objetivo 'Activity' en el conjunto de entrenamiento y testeo.
* Verificar la codificación de 'Activity' en las primeras 5 instancias de entrenamiento, así como que se hayan codificado correctamente todas las actividades presentes en el dataset.
* Escalar las variables numéricas.
* ¿Qué variables del dataset eligen como predictoras para su modelo y por qué?

In [ ]:
# Código

# 3. Reducción de dimensionalidad con PCA

El dataset contiene un número elevado de variables. Cuando trabajamos con muchas dimensiones, pueden aparecer varios desafíos: mayor costo computacional, redundancia entre variables, dificultad para interpretar los datos, o riesgo de sobreajuste en algunos modelos.

Para abordar este problema se utilizará **PCA (Análisis de Componentes Principales)**, una técnica que transforma las variables originales en un nuevo conjunto de componentes linealmente independientes que capturan gran parte de la variabilidad del sistema.

Aplicar PCA para reducir la dimensionalidad del dataset antes de entrenar.

### Guía orientadora

- ¿Cuánta varianza explican las primeras componentes?
- ¿Cómo se ve la varianza acumulada en función del número de componentes?
- ¿Cuántas componentes van a conservar?
- ¿Qué se gana y qué se pierde al reducir dimensionalidad?

In [ ]:
# Código

# 4.1 Modelo 1: Regresión Logística

Como primer modelo de referencia se utilizará una **Regresión Logística** para clasificación multiclase.

Aunque se trata de un modelo relativamente simple, suele ser una muy buena línea de base por varias razones:

- es rápido de entrenar,
- funciona bien en muchos problemas de clasificación,
- permite evaluar el impacto del preprocesamiento realizado,
- ofrece un punto de comparación frente a modelos más complejos.

En este caso se entrena sobre la representación reducida obtenida con **PCA**, lo cual ayuda a simplificar el espacio de entrada.

### Ajuste de hiperparámetros

No se utilizará una única configuración fija del modelo. En su lugar, se explorarán distintas combinaciones de hiperparámetros para encontrar una alternativa con mejor desempeño.

### Guía Orientadora
- Crear un modelo de Regresión Logística.
- Hacer un ajuste fino de hiperparámetros.
- Entrenar un modelo optimizado.

### Preguntas para pensar

- ¿Cómo influye la regularización en el modelo?
- ¿Qué ventaja puede tener usar PCA antes de una regresión logística?
- ¿Qué métricas y diagnósticos son adecuados para este problema?
- ¿El resultado obtenido parece adecuado como línea de base?


In [ ]:
# Código

# 4.2 Modelo 2: Árbol de Decisión

El segundo enfoque será un **Árbol de Decisión**.

A diferencia de la regresión logística, este tipo de modelo divide el espacio de variables mediante reglas del tipo:

- si una variable supera cierto umbral, ir hacia una rama,
- en caso contrario, ir hacia otra.

Esto lo vuelve un modelo muy interesante desde el punto de vista ingenieril porque:

- es relativamente fácil de interpretar,
- permite visualizar reglas de decisión,
- no requiere necesariamente escalado de variables,
- puede capturar relaciones no lineales.

### Nota importante

**En este entrenen el árbol sobre las variables originales (`X_train`), sin usar PCA.**
Esto tiene sentido porque los árboles no suelen beneficiarse del escalado del mismo modo que los modelos lineales, y además trabajar con variables originales facilita la interpretación del árbol y de la importancia de las características.

### Guía Orientadora
- Crear un modelo de Arbol de decisión. Trabajar con los datos sin PCA.
- Hacer un ajuste fino de algunos hiperparámetros del árbol.
- Entrenar un modelo optimizado.
- Evaluar la importancia de las variables predictoras.
- Visualizar el árbol de decisión.

### Preguntas para pensar

- ¿Qué ventajas ofrece este modelo respecto a la regresión logística?
- ¿Qué riesgos tiene un árbol demasiado profundo?
- ¿Qué información aporta la visualización del árbol?


In [ ]:
# Código

# 4.3 Modelo 3: Random Forest

Como tercer modelo se utilizará un **Random Forest**, que puede entenderse como un conjunto de muchos árboles de decisión entrenados de manera complementaria.

La idea principal detrás de este método es que, en lugar de confiar en un único árbol, se combinan múltiples árboles para obtener una predicción más robusta.

### Ventajas esperadas

Frente a un árbol individual, un bosque aleatorio suele:

- reducir el sobreajuste,
- mejorar la capacidad de generalización,
- ofrecer mejor desempeño predictivo en muchos problemas reales.

### Aspecto metodológico

Al igual que en el caso anterior, este modelo se entrena sobre las variables originales y no sobre la versión con PCA.  
Esto permite aprovechar la naturaleza del algoritmo y conservar interpretabilidad a través de la importancia de variables.

### Guía Orientadora
- Crear un modelo de Random Forest. Trabajar con los datos sin PCA.
- Hacer un ajuste fino de algunos hiperparámetros.
- Entrenar un modelo optimizado.
- Evaluar la importancia de las variables predictoras.

### Preguntas para pensar

- ¿Por qué un conjunto de árboles puede generalizar mejor que un solo árbol?
- ¿Qué se gana en desempeño y qué se pierde en interpretabilidad?

In [ ]:
# Código

Hasta este punto se entrenaron tres enfoques distintos:

Regresión Logística,
Árbol de Decisión,
Random Forest.

Ahora que entrenaron y optimizaron los tres modelos, pueden elegir el modelo con el mejor desempeño para evaluarlo finalmente con los datos de test.

El objetivo final no es solo obtener el modelo con el mejor resultado numérico, sino también analizar qué tipo de modelo resulta más adecuado para este problema según distintos criterios.

Preguntas de cierre
1. ¿Qué modelo obtuvo el mejor desempeño con los datos de entrenamiento?
2. ¿Cuál es el desempeño de ese modelo con los datos de test?
3. ¿Qué diferencias observaron entre las matrices de confusión de los distintos modelos?
4. ¿Qué rol jugó PCA en el enfoque lineal?
5. ¿Por qué los modelos basados en árboles se trabajaron sobre las variables originales?
6. Si tuvieran que desplegar uno de estos modelos en una aplicación real, ¿cuál elegirían y por qué?